<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/HES20GR004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# --- Imports & shared parameters ---
import numpy as np

N = 64
steps = 20
dt = 1e-2

chi   = 1.0
mu    = 0.10
beta  = 0.05
gamma = 0.12
beta_link = 0.08
kappa = 8.64e-4  # gentle back-pressure

rng = np.random.default_rng(42)
Phi0 = rng.normal(0, 1e-2, size=(N, N, N))

def l2_per_site(a):
    return np.linalg.norm(a) / a.size


In [6]:
# --- Finite-difference operators ---
def fd_lap(phi):
    return (np.roll(phi, -1, 0) + np.roll(phi, +1, 0) +
            np.roll(phi, -1, 1) + np.roll(phi, +1, 1) +
            np.roll(phi, -1, 2) + np.roll(phi, +1, 2) - 6*phi)

def fd_grad(phi, axis):
    return 0.5 * (np.roll(phi, -1, axis) - np.roll(phi, +1, axis))

# --- FD update rule (Grok-pure style) ---
def update_fd(Phi):
    lap = fd_lap(Phi)
    tanh_term = gamma * np.tanh(Phi)
    de_term   = kappa * Phi**4
    # link omitted for clarity; add if needed for your run
    return Phi + dt * (chi*lap + mu*tanh_term - beta*Phi + de_term)

# --- FD diagnostics ---
def diagnostics_fd(Phi):
    dpx = fd_grad(Phi, 0); dpy = fd_grad(Phi, 1); dpz = fd_grad(Phi, 2)
    lap = fd_lap(Phi)
    G00 = 2.0 * lap
    grad_sq = dpx**2 + dpy**2 + dpz**2
    T00 = grad_sq + kappa * (Phi**4)  # proxy to match Lite style
    coeff = 8.0 * np.pi
    R00 = l2_per_site(G00 - coeff*T00)
    B = l2_per_site(lap - G00/2.0)
    return R00, B

# --- Run FD track ---
Phi_fd = Phi0.copy()
log_fd = []
for s in range(steps):
    Phi_fd = update_fd(Phi_fd)
    R00, B = diagnostics_fd(Phi_fd)
    log_fd.append((R00, B))
print("FD track done.")


FD track done.


In [7]:
# --- Stronger filter + smaller timestep patch ---
dt = 5e-3   # smaller timestep for stability
eps = 5e-2  # stronger spectral filter

Phi_sp = Phi0.copy()
Phi_sf = Phi0.copy()

log_sp, log_sf = [], []
hk_sp, hk_sf = [], []

for s in range(steps):
    # FFT-only (still unstable, for comparison)
    Phi_sp = update_sp(Phi_sp, eps=0.0)
    R00, B = diagnostics_sp(Phi_sp)
    log_sp.append((R00, B))
    hk_sp.append(high_k_fraction(Phi_sp))

    # FFT+filter (covenant)
    Phi_sf = update_sp(Phi_sf, eps=eps)
    R00f, Bf = diagnostics_sp(Phi_sf)
    log_sf.append((R00f, Bf))
    hk_sf.append(high_k_fraction(Phi_sf))

print("Spectral tracks rerun with stronger filter and smaller dt.")


/tmp/ipython-input-807183477.py:42: RuntimeWarning: overflow encountered in power
  T00 = grad_sq + kappa * (Phi**4)  # same proxy as FD
/tmp/ipython-input-807183477.py:30: RuntimeWarning: overflow encountered in power
  de_term   = kappa * Phi**4
/tmp/ipython-input-807183477.py:11: RuntimeWarning: invalid value encountered in multiply
  dpx = np.fft.ifftn(1j*KX*phik).real
/tmp/ipython-input-807183477.py:12: RuntimeWarning: invalid value encountered in multiply
  dpy = np.fft.ifftn(1j*KY*phik).real
/tmp/ipython-input-807183477.py:13: RuntimeWarning: invalid value encountered in multiply
  dpz = np.fft.ifftn(1j*KZ*phik).real
/tmp/ipython-input-807183477.py:18: RuntimeWarning: invalid value encountered in multiply
  return np.fft.ifftn(-K2*phik).real


Spectral tracks rerun with stronger filter and smaller dt.


In [8]:
def pct(x0, x1):
    return 100.0 * (x0 - x1) / max(1e-12, x0)

print("step | FD_R00 | SP_R00 | SF_R00 | HK_SP | HK_SF | B_fd | B_sp | B_sf")
for s in range(steps):
    R_fd, B_fd = log_fd[s]
    R_sp, B_sp = log_sp[s]
    R_sf, B_sf = log_sf[s]
    print(f"{s:3d} | {R_fd:.3e} | {R_sp:.3e} | {R_sf:.3e} | {hk_sp[s]:.3f} | {hk_sf[s]:.3f} | {B_fd:.1e} | {B_sp:.1e} | {B_sf:.1e}")

# Quick verdicts
fd_drop = pct(log_fd[0][0], log_fd[-1][0])
sp_drop = pct(log_sp[0][0], log_sp[-1][0])
sf_drop = pct(log_sf[0][0], log_sf[-1][0])
print(f"\nResidual drop (%): FD={fd_drop:.1f}%, SP={sp_drop:.1f}%, SF={sf_drop:.1f}%")
print(f"High-k fraction (first→last): SP={hk_sp[0]:.3f}→{hk_sp[-1]:.3f}, SF={hk_sf[0]:.3f}→{hk_sf[-1]:.3f}")


step | FD_R00 | SP_R00 | SF_R00 | HK_SP | HK_SF | B_fd | B_sp | B_sf
  0 | 2.341e-04 | 1.792e+04 | 1.774e-01 | 0.937 | 0.667 | 0.0e+00 | 0.0e+00 | 0.0e+00
  1 | 2.162e-04 | 2.044e+09 | 1.767e-02 | 0.991 | 0.671 | 0.0e+00 | 0.0e+00 | 0.0e+00
  2 | 1.999e-04 | 6.624e+27 | 1.759e-03 | 0.415 | 0.601 | 0.0e+00 | 0.0e+00 | 0.0e+00
  3 | 1.850e-04 | 1.491e+110 | 1.752e-04 | 0.646 | 0.055 | 0.0e+00 | 0.0e+00 | 0.0e+00
  4 | 1.714e-04 | inf | 1.744e-05 | 0.661 | 0.001 | 0.0e+00 | 0.0e+00 | 0.0e+00
  5 | 1.590e-04 | nan | 1.737e-06 | nan | 0.000 | 0.0e+00 | nan | 0.0e+00
  6 | 1.476e-04 | nan | 1.730e-07 | nan | 0.000 | 0.0e+00 | nan | 0.0e+00
  7 | 1.371e-04 | nan | 1.722e-08 | nan | 0.000 | 0.0e+00 | nan | 0.0e+00
  8 | 1.276e-04 | nan | 1.717e-09 | nan | 0.000 | 0.0e+00 | nan | 0.0e+00
  9 | 1.188e-04 | nan | 1.722e-10 | nan | 0.000 | 0.0e+00 | nan | 0.0e+00
 10 | 1.108e-04 | nan | 1.799e-11 | nan | 0.000 | 0.0e+00 | nan | 0.0e+00
 11 | 1.034e-04 | nan | 2.321e-12 | nan | 0.000 | 0.0e+00 | na